# Cambridge, MA — 4:1 Split-Rate Land Value Tax Model (FY2026)

**Scope:** the full City of Cambridge property tax levy (in Massachusetts the municipal levy is the entire property tax — there is no separate county or school levy). Real property only; personal property (~$2.2B AV) is not on parcels.

**Reform:** 4:1 land-to-building rate ratio, revenue-neutral **within each tax class** (residential and commercial/industrial), so Cambridge's classified-rate shift and the residential exemption are preserved.

**Data:** City of Cambridge Assessing FY2026 Property Database + FY2026 parcel polygons, from the city's official GIS data repository (`github.com/cambridgegis/cambridgegis_data`).

**Official figures:** FY26 levy $678,852,471; residential rate $6.67, CIP rate $14.07 per $1,000; residential exemption $510,208; residential share of levy 33.8%; total taxable AV $73.11B (City of Cambridge FY26 Tax Rate Executive Summary and Tax Rate Letter, Oct 2025).

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'cambridge'
STATE_FIPS = '25'              # Massachusetts
COUNTY_FIPS = '017'            # Middlesex County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# Parcel-map export identity columns
PARCEL_ID_COL = 'PID'                  # Cambridge Assessing property ID (condo buildings: master record PID)
OWNER_NAME_COL = 'Owner_Name'
OWNER_ADDRESS_COL = 'Address'          # site address
PARCEL_URL_TEMPLATE = 'https://www.cambridgema.gov/PropertyDatabase/{parcel_id}'

# FY2026 official figures (City of Cambridge FY26 Tax Rate Executive Summary / Tax Rate Letter, Oct 2025)
RES_RATE = 6.67                    # $ per $1,000, residential class
CIP_RATE = 14.07                   # $ per $1,000, commercial / industrial / personal property
RESIDENTIAL_EXEMPTION = 510_208    # $ off assessed value for owner-occupied principal residences
OFFICIAL_LEVY = 678_852_471        # total FY26 levy (real + personal property)
OFFICIAL_RES_SHARE = 0.338         # residential share of the levy
OFFICIAL_TAXABLE_AV = 73_111_556_293  # total taxable AV incl. personal property

DATA_URL = 'https://raw.githubusercontent.com/cambridgegis/cambridgegis_data/main/Assessing/FY2026/FY2026_Parcels'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

In [2]:
# Step 2: Load the FY2026 property database and parcel polygons (Cambridge GIS GitHub mirror)
DB_PATH = DATA_DIR / 'ASSESSING_PropertyDatabase_FY2026.csv'
GEOM_PATH = DATA_DIR / 'ASSESSING_ParcelsFY2026.geojson'
for _path in [DB_PATH, GEOM_PATH]:
    if not _path.exists():
        import urllib.request
        urllib.request.urlretrieve(f'{DATA_URL}/{_path.name}', _path)
        print(f"Downloaded {_path.name}")

raw = pd.read_csv(DB_PATH, low_memory=False)
parcels_geo = gpd.read_file(GEOM_PATH)[['ML', 'geometry']]
print(f"Property database: {len(raw):,} rows, {raw['PID'].nunique():,} unique PIDs")
print(f"Parcel polygons: {len(parcels_geo):,}")

# Multi-building parcels repeat the parcel-level values once per BldgNum — keep one row per PID
db = raw.sort_values(['PID', 'BldgNum']).drop_duplicates('PID').copy()

# Condo master records (class 199 CONDO-BLDG) carry no values but do carry the building's lot area
condo_master = db[db['PropertyClass'] == 'CONDO-BLDG'][['PID', 'GISID', 'LandArea']].rename(
    columns={'PID': 'master_PID', 'LandArea': 'master_LandArea'})
condo_master = condo_master.drop_duplicates('GISID')
db = db[db['PropertyClass'] != 'CONDO-BLDG'].copy()
# A few records lack GISID; recover the map-lot from MapLot (first two dash-separated parts)
_ml = db['MapLot'].astype(str).str.split('-').str[:2].str.join('-')
db['GISID'] = db['GISID'].fillna(_ml)
print(f"Assessment records after de-duplication: {len(db):,} ({len(condo_master):,} condo master records set aside)")

for _col in ['LandValue', 'BuildingValue', 'AssessedValue', 'LandArea']:
    db[_col] = pd.to_numeric(db[_col], errors='coerce').fillna(0).clip(lower=0)

Property database: 30,156 rows, 28,850 unique PIDs
Parcel polygons: 12,933


Assessment records after de-duplication: 26,249 (2,601 condo master records set aside)


### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `LandValue` | Full market value (MA assesses at 100%) |
| Improvement value | `BuildingValue` | Buildings + structures |
| Total value | `AssessedValue` | Land + building |
| Use/class code | `StateClassCode` / `PropertyClass` | MA DOR state use codes (3-digit, optional 4th-digit sub-code) |
| Full exemption | `StateClassCode` 9xx | Government, education, religious, charitable, housing authority |
| Dollar exemption | `ResidentialExemption` (bool) | $510,208 off AV for owner-occupied residences (FY26) |
| Parcel ID | `PID` (record), `GISID` (map-lot, joins to polygons via `ML`) | |
| Tax district | `TaxDistrict` | Assessing neighborhoods (R* residential, C* commercial) |

**Assessment ratio:** 100% of full market value.
**Rates:** FY26 classified rates — residential $6.67, CIP $14.07 per $1,000 (City of Cambridge FY26 Tax Rate Letter).
The `PropertyTaxAmount` column in the open-data file is stale (identical across FY24–FY26 extracts), so it is **not** used;
current tax is rebuilt from the official FY26 rates.

In [3]:
# Step 3: Classify parcels

def _class3(code) -> str:
    """Normalize a MA state use code to its 3-digit base (4-digit codes carry a sub-code suffix)."""
    s = str(int(code))
    return s[:3] if len(s) == 4 else s.zfill(3)

db['class3'] = db['StateClassCode'].apply(_class3)
db['is_mixed_flag'] = db['PropertyClass'].str.startswith('MXD') | db['class3'].isin(['013', '014', '021', '031', '041'])

# Fully exempt: MA 9xx classes (government, education, charitable, religious, housing authority)
db['full_exmp'] = (db['class3'].str[0] == '9').astype(int)

# Tax class: CIP = 3xx/4xx (and primarily-commercial mixed-use 031/041); everything else taxable is residential.
# Cambridge splits a mixed-use parcel's value between classes by use percentage; the open-data file does not
# publish that split, so each mixed-use parcel is assigned to its primary class (01x/02x → residential, 03x/04x → CIP).
db['tax_class'] = np.where(
    db['full_exmp'] == 1, 'exempt',
    np.where(db['class3'].str[0].isin(['3', '4']) | db['class3'].isin(['031', '041']), 'CIP', 'RES'))

CONDO_CLASSES = {'102', '327', '339', '343', '409', '413'}
CONDO_SUBCODES = {1021, 1028, 1029}
db['is_condo_unit'] = (db['class3'].isin(CONDO_CLASSES) | db['StateClassCode'].isin(CONDO_SUBCODES)) & (db['full_exmp'] == 0)


def classify(row) -> str:
    c = row['class3']
    code = int(row['StateClassCode'])
    if row['is_mixed_flag']:
        return 'Mixed Use'
    if code in (1028,) or c in ('336', '337', '339') or code == 1067:
        return 'Transportation - Parking'
    if c == '102':
        return 'Condominium'
    if c == '101' or code == 1091:
        return 'Single Family Residential'
    if c in ('104', '105') or code in (1094, 1095):
        return 'Small Multi-Family (2-4 units)'
    if c in ('111', '112', '114') or code == 1098:
        return 'Large Multi-Family (5+ units)'
    if c in ('106', '130', '131', '132', '390', '391', '392', '440', '442') or code in (1061, 1062, 1322):
        return 'Vacant Land'
    if c in ('109', '113', '121', '122', '123'):
        return 'Other Residential'
    if c in ('300', '302'):
        return 'Hotel'
    if c in ('323', '324', '325', '326', '327'):
        return 'Retail / General Commercial'
    if c in ('340', '341', '342', '343', '345', '346'):
        return 'Office / Commercial Condo'
    if c in ('404', '406', '413'):
        return 'Commercial'          # lab / R&D / high-tech (MA classes these as industrial)
    if c in ('316', '400', '401', '407', '409', '422', '424', '430', '450'):
        return 'Industrial'
    if c[0] in ('3', '1'):
        return 'Other Commercial' if c[0] == '3' else 'Other Residential'
    return 'Other'

db['PROPERTY_CATEGORY'] = db.apply(classify, axis=1)
db.loc[db['full_exmp'] == 1, 'PROPERTY_CATEGORY'] = 'Exempt'
# Override: $0 building value → Vacant Land (condo units excluded — their land is imputed below)
db.loc[(db['BuildingValue'] <= 0) & (db['full_exmp'] == 0) & ~db['is_condo_unit'], 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(db.groupby('tax_class').agg(records=('PID', 'size'), assessed_value_B=('AssessedValue', lambda s: s.sum() / 1e9)).round(2))
print(db['PROPERTY_CATEGORY'].value_counts())
print("\nResidual buckets by source class:")
print(db[db['PROPERTY_CATEGORY'].isin(['Other', 'Other Residential', 'Other Commercial'])].groupby(['PROPERTY_CATEGORY', 'PropertyClass']).size())

           records  assessed_value_B
tax_class                           
CIP           1296             29.65
RES          24104             41.23
exempt         849             22.50
PROPERTY_CATEGORY
Condominium                       15446
Single Family Residential          3965
Small Multi-Family (2-4 units)     3428
Exempt                              849
Large Multi-Family (5+ units)       801
Office / Commercial Condo           451
Mixed Use                           320
Retail / General Commercial         302
Vacant Land                         245
Transportation - Parking            131
Commercial                          109
Industrial                           78
Other Commercial                     53
Other Residential                    48
Hotel                                23
Name: count, dtype: int64

Residual buckets by source class:
PROPERTY_CATEGORY  PropertyClass
Other Commercial   AUTO-REPAIR      14
                   AUTO-SALES        4
                   BILLBO

In [4]:
# Step 3b: Base reconciliation against the official FY26 taxable AV
parcel_taxable_av = db.loc[db['full_exmp'] == 0, 'AssessedValue'].sum()
implied_pp_av = OFFICIAL_TAXABLE_AV - parcel_taxable_av
print(f"Taxable real property AV in file: ${parcel_taxable_av/1e9:,.2f}B")
print(f"Official FY26 taxable AV (real + personal): ${OFFICIAL_TAXABLE_AV/1e9:,.2f}B")
print(f"Implied personal property AV: ${implied_pp_av/1e9:,.2f}B ({implied_pp_av/OFFICIAL_TAXABLE_AV*100:.1f}% of base — not on parcels)")

Taxable real property AV in file: $70.87B
Official FY26 taxable AV (real + personal): $73.11B
Implied personal property AV: $2.24B (3.1% of base — not on parcels)


### Condominium land imputation

Cambridge books every condominium unit (residential, office, retail, lab, parking) with **$0 land value** —
the land is folded into the unit's `BuildingValue`. Left as-is, a split-rate would treat ~15,000 condo units
as pure buildings and hand them a large, spurious tax cut. Following the St. Paul precedent, we:

1. Take the building's lot area from its condo master record (`CONDO-BLDG`, class 199), falling back to the GIS polygon area.
2. Price that lot at the **median land $/sq ft of non-condo parcels of the same tax class in the same assessing
   district** (`TaxDistrict`), falling back to the citywide class median.
3. Allocate the building's land to units in proportion to unit assessed value, and cap each unit's land at
   the 90th-percentile land share of non-condo parcels in its class (so a tiny lot can't absorb the whole unit).
4. Re-split each unit: land = imputed land, building = assessed value − land (total value unchanged).
5. After taxes are computed per unit, collapse units into one record per building (per tax class) for the
   export and map.

In [5]:
# Step 3c: Impute condo land

parcels_geo_area = parcels_geo.to_crs(2249)  # MA State Plane (ft)
parcels_geo['gis_sqft'] = parcels_geo_area.geometry.area
gis_sqft = parcels_geo.groupby('ML')['gis_sqft'].sum()

ref = db[(db['full_exmp'] == 0) & ~db['is_condo_unit'] & (db['LandValue'] > 0) & (db['LandArea'] > 0)
         & (db['BuildingValue'] > 0)].copy()
ref['land_psf'] = ref['LandValue'] / ref['LandArea']
ref['land_share'] = ref['LandValue'] / ref['AssessedValue']
psf_district = ref.groupby(['tax_class', 'TaxDistrict'])['land_psf'].median()
psf_city = ref.groupby('tax_class')['land_psf'].median()
share_cap = ref.groupby('tax_class')['land_share'].quantile(0.90)
print("Citywide median land $/sqft:", psf_city.round(0).to_dict())
print("Land-share cap (P90 of non-condo):", share_cap.round(3).to_dict())

condo = db[db['is_condo_unit']].copy()
condo = condo.merge(condo_master[['GISID', 'master_LandArea']], on='GISID', how='left')
condo['lot_sqft'] = condo['master_LandArea'].where(condo['master_LandArea'] > 0)
condo['lot_sqft'] = condo['lot_sqft'].fillna(condo['GISID'].map(gis_sqft))
condo['psf'] = [psf_district.get((tc, td), np.nan) for tc, td in zip(condo['tax_class'], condo['TaxDistrict'])]
condo['psf'] = condo['psf'].fillna(condo['tax_class'].map(psf_city))
bldg_av = condo.groupby('GISID')['AssessedValue'].transform('sum')
condo['imputed_land'] = condo['lot_sqft'] * condo['psf'] * condo['AssessedValue'] / bldg_av.replace(0, np.nan)
condo['imputed_land'] = np.minimum(condo['imputed_land'].fillna(0),
                                   condo['AssessedValue'] * condo['tax_class'].map(share_cap))
print(f"Condo units: {len(condo):,} in {condo['GISID'].nunique():,} buildings; "
      f"lot area missing for {condo['lot_sqft'].isna().sum():,} units")
print(f"Imputed condo land share: median {(condo['imputed_land']/condo['AssessedValue']).median():.1%}, "
      f"aggregate {condo['imputed_land'].sum()/condo['AssessedValue'].sum():.1%}")
print(f"Non-condo residential land share (aggregate): "
      f"{ref.loc[ref.tax_class=='RES','LandValue'].sum()/ref.loc[ref.tax_class=='RES','AssessedValue'].sum():.1%}")

db['land_model'] = db['LandValue']
db['bldg_model'] = db['BuildingValue']
db.loc[db['is_condo_unit'], 'land_model'] = condo['imputed_land'].values
db.loc[db['is_condo_unit'], 'bldg_model'] = (db.loc[db['is_condo_unit'], 'AssessedValue']
                                             - db.loc[db['is_condo_unit'], 'land_model']).clip(lower=0)

Citywide median land $/sqft: {'CIP': 170.0, 'RES': 236.0}
Land-share cap (P90 of non-condo): {'CIP': 0.914, 'RES': 0.629}


Condo units: 15,836 in 2,605 buildings; lot area missing for 40 units
Imputed condo land share: median 25.1%, aggregate 23.5%
Non-condo residential land share (aggregate): 40.5%


### Current tax (FY2026)

* Residential class: `(AV − residential exemption) × $6.67/1000`. The $510,208 residential exemption applies to
  owner-occupied principal residences (`ResidentialExemption == True`), is taken out of building value first, then land.
* CIP class: `AV × $14.07/1000`.
* 9xx classes are fully exempt (held out of the model).
* Personal property (≈$2.2B AV, ≈$31M of levy) is not on parcels and is excluded from the modeled levy.

Validation targets: residential levy = 33.8% × $678.85M = $229.5M; real-property levy = total levy − personal property.

In [6]:
# Step 4: Rebuild the current tax
db['res_exemption'] = np.where((db['tax_class'] == 'RES') & db['ResidentialExemption'].fillna(False).astype(bool),
                               RESIDENTIAL_EXEMPTION, 0.0)
db['millage_rate'] = np.select([db['tax_class'] == 'RES', db['tax_class'] == 'CIP'], [RES_RATE, CIP_RATE], 0.0)

# Exemption hierarchy: dollar exemption off building first, then land
_exm_bldg = np.minimum(db['res_exemption'], db['bldg_model'])
db['taxable_improvement_value'] = (db['bldg_model'] - _exm_bldg).clip(lower=0)
db['taxable_land_value'] = (db['land_model'] - (db['res_exemption'] - _exm_bldg)).clip(lower=0)
db.loc[db['full_exmp'] == 1, ['taxable_land_value', 'taxable_improvement_value']] = 0
db['taxable_value'] = db['taxable_land_value'] + db['taxable_improvement_value']

current_revenue, _, db = calculate_current_tax(
    df=db,
    tax_value_col='taxable_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

official_res = OFFICIAL_LEVY * OFFICIAL_RES_SHARE
implied_pp_tax = implied_pp_av * CIP_RATE / 1000
official_real = OFFICIAL_LEVY - implied_pp_tax
res_rev = db.loc[db['tax_class'] == 'RES', 'current_tax'].sum()
cip_rev = db.loc[db['tax_class'] == 'CIP', 'current_tax'].sum()
gap_res = (res_rev / official_res - 1) * 100
gap_pct = (current_revenue / official_real - 1) * 100
print(f"Residential: modeled ${res_rev:,.0f}   official ${official_res:,.0f}   gap {gap_res:+.2f}%")
print(f"CIP (real property): modeled ${cip_rev:,.0f}   official CIP levy less personal property ${OFFICIAL_LEVY - official_res - implied_pp_tax:,.0f}")
print(f"Modeled: ${current_revenue:,.0f}   Official (real property): ${official_real:,.0f}   Gap: {gap_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

Residential: modeled $233,199,408   official $229,452,135   gap +1.63%
CIP (real property): modeled $417,108,911   official CIP levy less personal property $417,930,925
Modeled: $650,308,318   Official (real property): $647,383,060   Gap: +0.45%


In [7]:
# Step 4b: Collapse condo units into one record per building (per tax class)
sum_cols = ['LandValue', 'BuildingValue', 'AssessedValue', 'land_model', 'bldg_model', 'taxable_land_value',
            'taxable_improvement_value', 'taxable_value', 'current_tax', 'res_exemption']
units = db[db['is_condo_unit']].copy()
units['_cat_av'] = units['AssessedValue']
agg = units.groupby(['GISID', 'tax_class']).agg(
    {**{c: 'sum' for c in sum_cols}, 'PID': 'first', 'Address': 'first', 'TaxDistrict': 'first',
     'millage_rate': 'first', 'full_exmp': 'first', 'is_condo_unit': 'first'})
agg['n_units'] = units.groupby(['GISID', 'tax_class']).size()
# Category = the category holding the most assessed value in the building/class group
_cat = units.groupby(['GISID', 'tax_class', 'PROPERTY_CATEGORY'])['_cat_av'].sum().reset_index()
_cat = _cat.sort_values('_cat_av').drop_duplicates(['GISID', 'tax_class'], keep='last').set_index(['GISID', 'tax_class'])
agg['PROPERTY_CATEGORY'] = _cat['PROPERTY_CATEGORY']
agg = agg.reset_index()
agg = agg.merge(condo_master[['GISID', 'master_PID']], on='GISID', how='left')
agg['PID'] = agg['master_PID'].fillna(agg['PID']).astype(int)
agg['Owner_Name'] = 'Condominium (' + agg['n_units'].astype(str) + ' units)'

gdf = pd.concat([db[~db['is_condo_unit']], agg.drop(columns='master_PID')], ignore_index=True, sort=False)
gdf['n_units'] = gdf['n_units'].fillna(1)

# Attach parcel polygons (map-lot GISID ↔ ML); a few map-lots have multiple polygon pieces → dissolve
_geo = parcels_geo.dissolve(by='ML').reset_index()[['ML', 'geometry']]
gdf = gpd.GeoDataFrame(gdf.merge(_geo, left_on='GISID', right_on='ML', how='left').drop(columns='ML'),
                       geometry='geometry', crs='EPSG:4326')
print(f"Records with parcel geometry: {gdf['geometry'].notna().mean():.1%}")
print(f"Modeling records: {len(gdf):,} ({len(agg):,} collapsed condo buildings from {len(units):,} units)")
print(f"Current tax preserved through collapse: ${gdf['current_tax'].sum():,.0f}")

Records with parcel geometry: 99.6%
Modeling records: 13,095 (2,682 collapsed condo buildings from 15,836 units)
Current tax preserved through collapse: $650,308,318


In [8]:
# Step 5: 4:1 split-rate, revenue-neutral within each tax class (preserves Cambridge's classification shift)
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

results = []
class_rates = {}
for _tc in ['RES', 'CIP']:
    _part = gdf[gdf['tax_class'] == _tc].copy()
    _lm, _im, _rev, _out = model_split_rate_tax(
        df=_part,
        land_value_col='taxable_land_value',
        improvement_value_col='taxable_improvement_value',
        current_revenue=_part['current_tax'].sum(),
        land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    )
    class_rates[_tc] = (_lm, _im, _rev)
    results.append(_out)
    print(f"{_tc}: land ${_lm:.3f}  building ${_im:.3f} per $1,000   (current {RES_RATE if _tc=='RES' else CIP_RATE})   revenue ${_rev:,.0f}")
gdf = gpd.GeoDataFrame(pd.concat(results).sort_index(), geometry='geometry', crs='EPSG:4326')

_total = sum(v[2] for v in class_rates.values())
land_millage = sum(v[0] * v[2] for v in class_rates.values()) / _total
improvement_millage = sum(v[1] * v[2] for v in class_rates.values()) / _total
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Revenue-weighted land millage: {land_millage:.4f}   improvement millage: {improvement_millage:.4f}")
print(f"Revenue check: current ${gdf['current_tax'].sum():,.0f}  new ${gdf['new_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

RES: land $11.910  building $2.978 per $1,000   (current 6.67)   revenue $233,199,408


CIP: land $31.550  building $7.888 per $1,000   (current 14.07)   revenue $417,108,911
Held out 849 fully-exempt parcels (excluded from model, export, and charts).
Revenue-weighted land millage: 24.5073   improvement millage: 6.1268
Revenue check: current $650,308,318  new $650,308,318

cambridge — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential   3965      $6,286,451       11.2%     $1,585       $1,864   26.9%      21.4%            65.0%             9.8%
Small Multi-Family (2-4 units)   3428      $4,666,686       12.6%     $1,361       $1,509   19.8%      15.8%            60.0%            10.2%
                   Condominium   2583     $-8,705,413      -11.3%    $-3,370         $725   11.2%       6.0%            44.9%            26.2%
 Large Multi-Family (5+ units)    801     $-2,555,758       -5.9%    $-3,191       $1,329    9.5%     

In [9]:
# Step 5b: Artifact scan (validate.md Gate 5)
_d = gdf.copy()
_d['bldg_share'] = _d['bldg_model'] / (_d['land_model'] + _d['bldg_model']).replace(0, np.nan)
scan = _d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('bldg_model', lambda s: (s <= 1000).mean()),
    current_tax_M=('current_tax', lambda s: s.sum() / 1e6),
).sort_values('median_pct', ascending=False)
print(scan.round(3))

                                   n  median_pct  median_bldg_share  \
PROPERTY_CATEGORY                                                     
Transportation - Parking          84     122.114              0.013   
Industrial                        76      88.453              0.213   
Other Commercial                  53      80.410              0.261   
Vacant Land                      245      78.563              0.000   
Retail / General Commercial      225      34.751              0.532   
Hotel                             23      27.361              0.576   
Office / Commercial Condo        301      23.376              0.600   
Single Family Residential       3965      21.382              0.544   
Other Residential                 48      17.951              0.460   
Mixed Use                        317      17.323              0.538   
Small Multi-Family (2-4 units)  3428      15.803              0.560   
Large Multi-Family (5+ units)    801       7.499              0.541   
Condom

### Reading the results (Gate 5)

* **Revenue neutrality is per class.** Residential and CIP are solved separately, so Cambridge's classification
  shift (CIP pays ~2.1× the residential rate) is preserved; the reform only moves tax from buildings to land *within* each class.
  Residential: land ≈ $11.9 / building ≈ $3.0 per $1,000 (from $6.67). CIP: land ≈ $31.6 / building ≈ $7.9 (from $14.07).
* **Lab / R&D ("Commercial") falls ~15%** while office, retail and hotels rise: Kendall Square lab buildings are
  extremely building-heavy (median land share ~17%), so they are the main CIP beneficiaries. Real result, not an artifact.
* **Parking, Industrial and Other Commercial rise sharply.** These are land-heavy: surface/structured parking, power and
  utility plants (machinery is personal property, so the real-property record is mostly land), gas stations, auto repair.
* **Single-family homes rise (~+21% median), two/three-families ~+16%.** Within the residential class the building-heavy
  stock is condominiums and large apartment buildings, so detached homes on Cambridge's expensive lots carry more of the
  residential levy. Two modeling choices drive the size of this number:
  1. *Residential exemption hierarchy* — the $510,208 exemption is taken off building value first (the LVTShift convention),
     which leaves owner-occupied homes' taxable value land-heavy. Applying it pro rata to land and building gives a
     single-family median of ~+14%.
  2. *Condo land imputation* — pricing condo lots at neighborhood land $/sq ft gives condos a ~24% aggregate land share.
     Imputing instead from the neighborhood median land *share* of non-condo parcels (~40%) gives a single-family
     median of ~+10% and a condo median of ~+14%.
  The lot-area method is kept as primary because it values condo land the same way the assessor values neighboring lots.
* **Condominium buildings** (collapsed from units): median +6%, but −11% in aggregate — large, dense towers gain the
  most; small condo conversions on big lots rise.

In [10]:
# Step 6: Exploration — median change by category
fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median()
summary.sort_values().plot.barh(ax=ax, color=['#2E7D32' if v < 0 else '#B3261E' for v in summary.sort_values()])
ax.set_title(f'{CITY_NAME.title()} — Median Tax Change % by Category (4:1 Split-Rate, per class)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

### Census join + standard export

The census join is optional and runs only when `CENSUS_API_KEY` is configured; without it the three core
tax-impact charts are still produced and the four neighborhood-equity charts are skipped (expected, not a failure).

In [11]:
# Census join — OPTIONAL, must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                # census_gdf already carries demographics — do NOT merge census_data a second time
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

Census join: skipped (no CENSUS_API_KEY)


In [12]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + self-contained interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

  ✓ cambridge: 12,246 rows → ../../analysis/data/cambridge.csv  [model: split_rate_4to1]


create_city_report: excluded 39 fully-exempt parcels (12,246 → 12,207 modeled).


  ✓ cambridge: 12,216 parcels → ../../analysis/maps/cambridge.parquet  [map export; 12,216 with record links]


  ✓ cambridge: interactive map → ../../analysis/reports/cambridge/parcel_map.html  [12,216 parcels]
Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | +0.45% vs. official real-property levy $647.4M (FY26 levy $678.85M less implied personal property ≈$31.5M); residential class +1.6% vs. $229.5M (33.8% share) |
| Base reconciliation | Taxable real property $70.87B vs. official total taxable $73.11B → implied personal property $2.24B (3.1%), consistent with personal property not being on parcels |
| Parcel count | 12,246 modeled records (from 26,249 assessment records: 15,836 condo units collapsed into 2,682 buildings); 849 fully-exempt (9xx) parcels held out and excluded from charts; 39 residential parcels whose AV is fully covered by the residential exemption owe $0 and are dropped by the report |
| Census coverage | N/A — skipped (no `CENSUS_API_KEY`); the four equity charts are not produced |
| PNGs generated | 3 of 3 core charts (equity charts skipped) |
| Parcel map | `analysis/maps/cambridge.parquet` (12,216 parcels with geometry) + `parcel_map.html`; links to `cambridgema.gov/PropertyDatabase/{PID}` (condo buildings link to the condo master record) |
| Artifact scan (Gate 5) | Condo $0-land artifact found and fixed (lot-area land imputation + building collapse). No ceiling clustering; building shares plausible (lab/R&D ~83%, parking/industrial land-heavy) |
| SFR median change | +21.4% (sensitivity: +14% with pro-rata residential exemption; +10% with share-based condo land imputation) |
| Vacant land median change | +78.6% |

### Limitations
* Mixed-use parcels are assigned wholly to their primary class; Cambridge actually splits their value between residential and CIP rates by use percentage (not published).
* Condo land is imputed (see above) — the single largest judgment call for the residential results.
* The open-data `PropertyTaxAmount` field is stale and is not used; FY26 tax is rebuilt from official rates, so abatements and small statutory exemptions (elderly, veteran, blind clauses) are not modeled.
* Personal property (~$31.5M of levy) is outside the model and is assumed to stay at the current CIP rate.
* 9xx exempt classes include a few PILOT-type parcels with small observed tax; all are treated as fully exempt.